# Compare CNN and FNO Runs

This notebook compares two trained model runs from `logs/<run_name>/`.

It compares:
- training and validation loss curves
- validation/test/OOD metrics
- relative improvement between models
- optional prediction/error plots from the best checkpoints

In [ ]:
from pathlib import Path
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("default")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

RUNS = {
    "CNN": "cnn_local_baseline_50ep",
    "FNO": "fno_local_50ep",
}

LOG_DIR = ROOT / "logs"
CHECKPOINT_DIR = ROOT / "checkpoints"
FIGURE_DIR = ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

print(f"Repo root: {ROOT}")
for label, run_name in RUNS.items():
    print(f"{label}: {run_name}")

## Load Run Logs

In [ ]:
def read_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def read_history(path):
    with path.open("r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise ValueError(f"Empty history file: {path}")
    history = {}
    for key in rows[0].keys():
        values = [row[key] for row in rows]
        history[key] = np.array(values, dtype=int if key == "epoch" else float)
    return history

runs = {}
for label, run_name in RUNS.items():
    run_dir = LOG_DIR / run_name
    if not run_dir.exists():
        raise FileNotFoundError(f"Missing run directory: {run_dir}")
    runs[label] = {
        "run_name": run_name,
        "config": read_json(run_dir / "config.json"),
        "metrics": read_json(run_dir / "metrics.json"),
        "history": read_history(run_dir / "history.csv"),
        "checkpoint": CHECKPOINT_DIR / f"{run_name}_best.pt",
    }

for label, run in runs.items():
    cfg = run["config"]
    met = run["metrics"]
    print(f"\n{label} ({run['run_name']})")
    print(f"  model:      {cfg['model']}")
    print(f"  epochs:     {cfg['epochs']}")
    print(f"  best epoch: {met['best_epoch']}")
    print(f"  params:     {cfg['parameter_count']:,}")
    print(f"  device:     {cfg['device']}")

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), constrained_layout=True)

for label, run in runs.items():
    hist = run["history"]
    epochs = hist["epoch"]
    axes[0].plot(epochs, hist["train_loss"], label=f"{label} train")
    axes[0].plot(epochs, hist["val_loss"], linestyle="--", label=f"{label} val")
    axes[1].plot(epochs, hist["val_rel_l2"], marker="o", markersize=3, label=label)
    axes[2].plot(epochs, hist["val_max_abs"], marker="o", markersize=3, label=label)

axes[0].set_title("Train/val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Normalized MSE loss")
axes[0].set_yscale("log")
axes[0].legend(frameon=False, fontsize=8)

axes[1].set_title("Validation relative L2")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Relative L2")
axes[1].legend(frameon=False)

axes[2].set_title("Validation max abs error")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Mean max abs error")
axes[2].legend(frameon=False)

out_path = FIGURE_DIR / "model_comparison_training_curves.png"
fig.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved {out_path}")

## Final Metrics by Split

In [ ]:
splits = ["val", "test", "ood_corner", "ood_multi"]
metric_names = ["mse", "rel_l2", "max_abs"]

for split in splits:
    print(f"\n{split}")
    for label, run in runs.items():
        vals = run["metrics"][split]
        print(
            f"  {label:>4s}: "
            f"mse={vals['mse']:.6g}, "
            f"rel_l2={vals['rel_l2']:.5f}, "
            f"max_abs={vals['max_abs']:.5f}"
        )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
x = np.arange(len(splits))
width = 0.36
labels = list(runs.keys())

for ax, metric in zip(axes, metric_names):
    for i, label in enumerate(labels):
        values = [runs[label]["metrics"][split][metric] for split in splits]
        offset = (i - (len(labels) - 1) / 2) * width
        ax.bar(x + offset, values, width=width, label=label)
    ax.set_title(metric)
    ax.set_xticks(x)
    ax.set_xticklabels(splits, rotation=25, ha="right")
    ax.set_ylabel(metric)
    ax.legend(frameon=False)

out_path = FIGURE_DIR / "model_comparison_final_metrics.png"
fig.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved {out_path}")

## Relative Improvement

Positive values mean the second model listed in `RUNS` has lower error than the first model.

In [ ]:
baseline_label, candidate_label = labels[0], labels[1]

improvements = {}
for split in splits:
    improvements[split] = {}
    for metric in metric_names:
        baseline = runs[baseline_label]["metrics"][split][metric]
        candidate = runs[candidate_label]["metrics"][split][metric]
        improvements[split][metric] = 100.0 * (baseline - candidate) / baseline

for split in splits:
    print(f"\n{split}: {candidate_label} vs {baseline_label}")
    for metric in metric_names:
        print(f"  {metric:>7s}: {improvements[split][metric]:>7.2f}%")

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
rel_l2_improvement = [improvements[split]["rel_l2"] for split in splits]
colors = ["tab:green" if value >= 0 else "tab:red" for value in rel_l2_improvement]
ax.bar(splits, rel_l2_improvement, color=colors)
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"Relative L2 improvement: {candidate_label} vs {baseline_label}")
ax.set_ylabel("Improvement (%)")
ax.set_xticklabels(splits, rotation=25, ha="right")

out_path = FIGURE_DIR / "model_comparison_rel_l2_improvement.png"
fig.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved {out_path}")

## Save Summary Tables

In [ ]:
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

summary_rows = []
for label, run in runs.items():
    cfg = run["config"]
    met = run["metrics"]
    for split in splits:
        row = {
            "label": label,
            "run_name": run["run_name"],
            "model": cfg["model"],
            "split": split,
            "best_epoch": met["best_epoch"],
            "parameter_count": cfg["parameter_count"],
            "mse": met[split]["mse"],
            "rel_l2": met[split]["rel_l2"],
            "max_abs": met[split]["max_abs"],
        }
        summary_rows.append(row)

csv_path = RESULTS_DIR / "model_comparison_summary.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

json_path = RESULTS_DIR / "model_comparison_summary.json"
payload = {
    "runs": RUNS,
    "summary": summary_rows,
    "improvements_percent": improvements,
}
with json_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print(f"Saved {csv_path}")
print(f"Saved {json_path}")

## Optional: Prediction Comparison from Checkpoints

This section loads each best checkpoint and plots ground truth, prediction, and absolute error for one sample. It requires PyTorch in the notebook kernel.

In [ ]:
try:
    import torch
    from dataset import HeatDataset, NormalizationStats, load_normalization_stats
    from models import HeatCNN, HeatFNO
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    print(f"Skipping prediction comparison because PyTorch import failed: {exc}")

TORCH_AVAILABLE

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def stats_from_checkpoint(ckpt, fallback):
    payload = ckpt.get("normalization_stats")
    if payload is None:
        return fallback
    return NormalizationStats(
        x_mean=payload["x_mean"].float().cpu(),
        x_std=payload["x_std"].float().cpu(),
        y_mean=payload["y_mean"].float().cpu(),
        y_std=payload["y_std"].float().cpu(),
    )

def make_model_from_config(config):
    if config["model"] == "cnn":
        return HeatCNN(
            hidden_channels=config.get("hidden_channels", 64),
            depth=config.get("depth", 5),
            residual_initial=not config.get("no_residual_initial", False),
        )
    if config["model"] == "fno":
        return HeatFNO(
            width=config.get("fno_width", 32),
            modes1=config.get("fno_modes", 12),
            modes2=config.get("fno_modes", 12),
            depth=config.get("fno_depth", 4),
            use_grid=not config.get("fno_no_grid", False),
            residual_initial=not config.get("no_residual_initial", False),
        )
    raise ValueError(f"Unknown model: {config['model']}")

def denormalize_y(y, stats):
    return y * stats.y_std.to(y.device) + stats.y_mean.to(y.device)

In [ ]:
SPLIT = "test"
SAMPLE_IDX = 0

if TORCH_AVAILABLE:
    device = get_device()
    fallback_stats = load_normalization_stats(ROOT / "data")
    loaded = {}
    for label, run in runs.items():
        ckpt = torch.load(run["checkpoint"], map_location=device)
        stats = stats_from_checkpoint(ckpt, fallback_stats)
        stats_device = NormalizationStats(
            x_mean=stats.x_mean.to(device),
            x_std=stats.x_std.to(device),
            y_mean=stats.y_mean.to(device),
            y_std=stats.y_std.to(device),
        )
        model = make_model_from_config(run["config"]).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()
        loaded[label] = {"model": model, "stats": stats, "stats_device": stats_device}
    print(f"Loaded checkpoints on {device}")

In [ ]:
if TORCH_AVAILABLE:
    # Use the first run's normalization stats for the sample input. The stats should match
    # because both models were trained on the same train split.
    first_stats = next(iter(loaded.values()))["stats"]
    first_stats = NormalizationStats(
        x_mean=first_stats.x_mean.cpu(),
        x_std=first_stats.x_std.cpu(),
        y_mean=first_stats.y_mean.cpu(),
        y_std=first_stats.y_std.cpu(),
    )
    dataset = HeatDataset(ROOT / "data" / f"{SPLIT}.npz", stats=first_stats, normalize=True)
    sample = dataset[SAMPLE_IDX]
    x = sample["x"].unsqueeze(0).to(device)
    y_true = dataset.y[SAMPLE_IDX, 0].numpy()
    initial = dataset.x[SAMPLE_IDX, 0].numpy()
    heater = dataset.x[SAMPLE_IDX, 1].numpy()
    alpha = float(dataset.alpha[SAMPLE_IDX])

    predictions = {}
    with torch.no_grad():
        for label, bundle in loaded.items():
            pred_norm = bundle["model"](x)
            pred = denormalize_y(pred_norm, bundle["stats_device"]).squeeze().cpu().numpy()
            predictions[label] = pred

    panels = [(y_true, "Ground truth", "inferno")]
    for label, pred in predictions.items():
        panels.append((pred, f"{label} pred", "inferno"))
        panels.append((np.abs(pred - y_true), f"{label} abs err", "viridis"))

    fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3), constrained_layout=True)
    for ax, (field, title, cmap) in zip(axes, panels):
        im = ax.imshow(field, origin="lower", cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(f"{SPLIT} sample {SAMPLE_IDX} | alpha = {alpha:.4f}", y=1.05)
    out_path = FIGURE_DIR / "model_comparison_predictions.png"
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_path}")